In [1]:
import numpy as np
import pandas as pd
import time
import logging
import sys
import json
from datetime import datetime
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
from scipy.stats import skew
import xgboost as xgb
from xgboost import XGBRegressor
import joblib

# --- 1. 配置与环境设置 ---
def setup_logging(log_file='training_housing_xgb.log'):
    log_formatter = logging.Formatter("%(asctime)s [%(levelname)-5.5s]  %(message)s")
    root_logger = logging.getLogger()
    root_logger.handlers = []
    root_logger.setLevel(logging.INFO)
    
    file_handler = logging.FileHandler(log_file, mode='w')
    file_handler.setFormatter(log_formatter)
    root_logger.addHandler(file_handler)
    
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(log_formatter)
    root_logger.addHandler(console_handler)

np.random.seed(42)

# --- 2. 数据准备 ---
def prepare_housing_data():
    """准备Ames Housing房价预测数据集"""
    logging.info("正在准备Ames Housing房价预测数据集...")

    # 加载数据
    train_df = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
    test_df = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

    # 合并数据集
    full_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)

    # 缺失值处理
    fill_none = ["Alley", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
                 "FireplaceQu", "GarageType", "GarageFinish", "GarageQual", "GarageCond", "PoolQC",
                 "Fence", "MiscFeature", "MasVnrType"]
    for col in fill_none:
        if col in full_df.columns:
            full_df[col] = full_df[col].fillna("None")
    
    fill_zero = ["BsmtFullBath", "BsmtHalfBath", "BsmtUnfSF", "GarageArea", "GarageCars", "MasVnrArea",
                 "PoolArea", "MiscVal", "OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch",
                 "TotRmsAbvGrd", "BedroomAbvGr", "Fireplaces", "HalfBath", "FullBath", "KitchenAbvGr",
                 "LotFrontage", "WoodDeckSF"]
    for col in fill_zero:
        if col in full_df.columns:
            full_df[col] = full_df[col].fillna(0)
    
    # 其他数值型特征用中位数填充
    num_cols = full_df.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        if full_df[col].isnull().sum() > 0:
            full_df[col] = full_df[col].fillna(full_df[col].median())

    # 删除无用特征
    cols_to_drop = ['Id', 'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 
                    'LotFrontage', 'GarageYrBlt', 'MoSold', 'YrSold']
    full_df = full_df.drop(cols_to_drop, axis=1, errors='ignore')

    # 特征工程
    full_df["TotalBath"] = full_df["BsmtFullBath"] + 0.5 * full_df["BsmtHalfBath"] + full_df["FullBath"] + 0.5 * full_df["HalfBath"]
    full_df["AllSF"] = full_df["GrLivArea"] + full_df["TotalBsmtSF"]
    full_df["AllFlrsSF"] = full_df["1stFlrSF"] + full_df["2ndFlrSF"]
    full_df["AllPorchSF"] = full_df["OpenPorchSF"] + full_df["EnclosedPorch"] + full_df["3SsnPorch"] + full_df["ScreenPorch"]

    # 类别型特征独热编码
    categorical_cols = full_df.select_dtypes(include=["object"]).columns
    full_df = pd.get_dummies(full_df, columns=categorical_cols)

    # 数值型特征对偏态进行log1p变换（绝对偏度>0.5）
    numeric_feats = full_df.select_dtypes(include=[np.number]).columns
    skewed_feats = full_df[numeric_feats].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
    skewness = skewed_feats[abs(skewed_feats) > 0.5]
    for feat in skewness.index:
        full_df[feat] = np.log1p(full_df[feat])

    # 分离训练集和测试集
    train_df = full_df.iloc[:len(train_df), :]
    test_df = full_df.iloc[len(train_df):, :].drop(['SalePrice'], axis=1)

    # 特征和目标变量
    X = train_df.drop(['SalePrice'], axis=1)
    y = train_df['SalePrice']

    # 目标变量log1p变换
    y = np.log1p(y)

    # 划分训练验证集
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    logging.info("Ames Housing房价预测数据集准备完毕。")
    logging.info(f"训练样本数: {len(X_train)}, 验证样本数: {len(X_val)}")
    logging.info(f"特征维度: {X_train.shape[1]}, 目标维度: 1")

    return X_train, X_val, y_train, y_val, X.columns

# --- 3. XGBoost模型训练与参数搜索 ---
def train_xgboost_with_search(X_train, y_train, X_val, y_val, feature_names):
    """训练XGBoost模型并进行参数搜索"""
    logging.info("="*30)
    logging.info("  启动XGBoost房价预测训练流程")
    logging.info("="*30)
    
    # 基本参数
    params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'seed': 42,
        'n_jobs': -1
    }
    
    # 参数搜索空间
    # param_grid = {
    #     'learning_rate': [0.01, 0.05, 0.1],
    #     'max_depth': [3, 5, 7],
    #     'min_child_weight': [1, 3, 5],
    #     'subsample': [0.7, 0.9],
    #     'colsample_bytree': [0.7, 0.9],
    #     'n_estimators': [100, 200, 300],
    #     'gamma': [0, 0.1, 0.2],
    #     'reg_alpha': [0, 0.1, 1],
    #     'reg_lambda': [0, 0.1, 1]
    # }
    param_grid = {
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5],
    'n_estimators': [100, 200]
}
    # 创建模型
    xgb_model = XGBRegressor(**params)
    
    # 网格搜索
    grid_search = GridSearchCV(
        estimator=xgb_model,
        param_grid=param_grid,
        cv=3,
        scoring='neg_mean_squared_error',  # 使用MSE作为评分指标
        verbose=1,
        n_jobs=-1
    )
    
    logging.info("开始参数搜索...")
    start_time = time.time()
    
    grid_search.fit(X_train, y_train)
    
    logging.info(f"参数搜索完成，耗时: {time.time()-start_time:.2f}秒")
    logging.info(f"最佳参数: {grid_search.best_params_}")
    logging.info(f"最佳验证分数(MSE): {-grid_search.best_score_:.4f}")
    
    # 使用最佳参数重新训练模型
    best_model = grid_search.best_estimator_
    
    # 在验证集上评估
    val_pred = best_model.predict(X_val)
    val_mse = mean_squared_error(y_val, val_pred)
    logging.info(f"最佳模型在验证集上的MSE: {val_mse:.4f}")
    logging.info(f"最佳模型在验证集上的RMSE: {np.sqrt(val_mse):.4f}")
    
    # 特征重要性
    importance = best_model.feature_importances_
    feature_importance = sorted(zip(feature_names, importance), key=lambda x: x[1], reverse=True)[:20]
    logging.info("\nTop 20重要特征:")
    for feat, imp in feature_importance:
        logging.info(f"{feat}: {imp:.4f}")
    
    return best_model, val_mse

# --- 4. 主程序 ---
if __name__ == '__main__':
    # 文件与超参数定义
    LOG_FILE = 'training_housing_xgb.log'
    JSON_LOG_FILE = 'training_housing_xgb_log.json'
    MODEL_SAVE_PATH = 'housing_xgb.pkl'
    
    setup_logging(LOG_FILE)
    
    # 准备数据
    X_train, X_val, y_train, y_val, feature_names = prepare_housing_data()
    
    # 训练模型
    start_time = time.time()
    best_model, best_val_mse = train_xgboost_with_search(X_train, y_train, X_val, y_val, feature_names)
    total_time = time.time() - start_time
    
    logging.info(f"总训练时间: {total_time:.2f}秒")
    
    # 保存模型
    joblib.dump(best_model, MODEL_SAVE_PATH)
    logging.info(f"模型已保存到 {MODEL_SAVE_PATH}")
    
    # 保存日志
    log_data = {
        "experiment_timestamp": datetime.now().isoformat(),
        "best_val_mse": best_val_mse,
        "best_val_rmse": np.sqrt(best_val_mse),
        "training_time": total_time,
        "top_features": {feat: float(imp) for feat, imp in zip(feature_names, best_model.feature_importances_) 
                        if imp > 0}  # 修正特征重要性保存方式
    }
    
    with open(JSON_LOG_FILE, 'w') as f:
        json.dump(log_data, f, indent=4)
    
    # 验证保存的模型
    loaded_model = joblib.load(MODEL_SAVE_PATH)
    val_pred = loaded_model.predict(X_val)
    loaded_mse = mean_squared_error(y_val, val_pred)
    
    logging.info("\n" + "="*30)
    logging.info("  模型验证结果")
    logging.info("="*30)
    logging.info(f"原始验证MSE: {best_val_mse:.4f}")
    logging.info(f"加载后验证MSE: {loaded_mse:.4f}")
    
    if abs(loaded_mse - best_val_mse) < 1e-3:
        logging.info("验证成功！模型可以被正确保存和复用。")
    else:
        logging.warning(f"验证损失差异: {abs(loaded_mse - best_val_mse):.4f}")

2025-08-21 12:37:20,578 [INFO ]  正在准备Ames Housing房价预测数据集...
2025-08-21 12:37:20,891 [INFO ]  Ames Housing房价预测数据集准备完毕。
2025-08-21 12:37:20,892 [INFO ]  训练样本数: 1168, 验证样本数: 292
2025-08-21 12:37:20,893 [INFO ]  特征维度: 279, 目标维度: 1
2025-08-21 12:37:20,894 [INFO ]  ==============================
2025-08-21 12:37:20,895 [INFO ]    启动XGBoost房价预测训练流程
2025-08-21 12:37:20,896 [INFO ]  ==============================
2025-08-21 12:37:20,896 [INFO ]  开始参数搜索...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
2025-08-21 12:37:29,159 [INFO ]  参数搜索完成，耗时: 8.26秒
2025-08-21 12:37:29,160 [INFO ]  最佳参数: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}
2025-08-21 12:37:29,161 [INFO ]  最佳验证分数(MSE): 0.0001
2025-08-21 12:37:29,190 [INFO ]  最佳模型在验证集上的MSE: 0.0001
2025-08-21 12:37:29,191 [INFO ]  最佳模型在验证集上的RMSE: 0.0107
2025-08-21 12:37:29,193 [INFO ]  
Top 20重要特征:
2025-08-21 12:37:29,193 [INFO ]  OverallQual: 0.1577
2025-08-21 12:37:29,194 [INFO ]  AllSF: 0.1191
2025-08-21 12:37:29,194 [INFO 